# Scaling laws: tự đo trên GPT mini, rồi đọc Chinchilla

Bạn train một loạt GPT mini chỉ khác `n_embd` trên cùng dữ liệu và cùng số token, khớp một **power law** cho loss theo số
tham số, rồi dùng công thức của Chinchilla để trả lời câu hỏi mà mọi đội pretrain phải trả lời: với một ngân sách compute,
nên chọn model bao lớn và train trên bao nhiêu token.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật, train, vẽ; test bỏ qua chúng.
- Mặc định chạy trên CPU (5 model nhỏ, khoảng 6 phút); kernel `ai-gpu` tự chọn model và số bước lớn hơn.
- Chấm: `uv run pytest tracks/ai-engineer/modules/05-transformers-llm-architecture/exercises/intermediate/03-scaling-laws -q`

In [ ]:
# explore: giới hạn luồng, tìm gốc repo, kiểm Wikipedia tiếng Việt
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
WIKI = ROOT / "data/raw/wikipedia-vi"
if not (WIKI.is_dir() and any(WIKI.iterdir())):
    sys.path.insert(0, str(ROOT))
    from data.download_dataset import ensure

    ensure("wikipedia-vi")
SHARD = WIKI / "20231101.vi/train-00002-of-00004.parquet"

In [ ]:
import math

import numpy as np

## 1. Đếm tham số và compute

- `gpt_params(n_layer, n_embd, vocab_size, block_size) -> dict`: số tham số của lớp `GPT` bài Basic 01 (bias đầy đủ, weight
  tying), trả `{"total": ..., "non_embedding": ...}`; `non_embedding` trừ `tok_emb` và `pos_emb` (đúng như
  `GPT.num_params(non_embedding=True)`). Suy công thức từ cấu trúc một Block, không khởi tạo model.
- `train_flops(n_params, n_tokens)`: xấp xỉ $$C \approx 6ND$$ FLOP cho một lần train (forward $$2N$$, backward $$4N$$ mỗi token).

In [ ]:
def gpt_params(n_layer: int, n_embd: int, vocab_size: int, block_size: int) -> dict[str, int]:
    d = n_embd
    non_embedding = n_layer * (12 * d * d + 13 * d) + 2 * d
    return {"total": non_embedding + vocab_size * d + block_size * d, "non_embedding": non_embedding}


def train_flops(n_params: float, n_tokens: float) -> float:
    return 6.0 * n_params * n_tokens

## 2. Khớp power law và công thức Chinchilla

- `fit_power_law(x, y) -> (a, alpha, c)`: khớp $$y \approx c + a x^{-\alpha}$$ với $$a > 0$$, $$\alpha > 0$$,
  $$0 \le c < \min y$$ ($$c$$ là loss "không giảm được"). Gợi ý cách làm đủ tốt: dò $$c$$ trên một lưới, với mỗi $$c$$ hồi
  quy tuyến tính $$\log(y - c)$$ theo $$\log x$$, giữ $$c$$ cho sai số bình phương **trên $$y$$** nhỏ nhất. Mảng numpy vào,
  ba số `float` ra.
- `chinchilla_loss(n_params, n_tokens)`: công thức khớp của Hoffmann và cộng sự (2022, cách 3):
  $$L(N, D) = E + \frac{A}{N^{\alpha}} + \frac{B}{D^{\beta}}$$ với $$E = 1.69, A = 406.4, B = 410.7, \alpha = 0.34, \beta = 0.28$$.
- `chinchilla_optimal(compute, tokens_per_param=20.0) -> (n_params, n_tokens)`: chia ngân sách $$C = 6ND$$ theo quy tắc
  $$D = 20N$$ của Chinchilla.

In [ ]:
def fit_power_law(x: np.ndarray, y: np.ndarray) -> tuple[float, float, float]:
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    best = None
    for c in np.linspace(0.0, y.min() * 0.999, 2000):
        slope, intercept = np.polyfit(np.log(x), np.log(y - c), 1)
        a, alpha = math.exp(intercept), -slope
        err = float(np.sum((c + a * x ** -alpha - y) ** 2))
        if best is None or err < best[0]:
            best = (err, a, alpha, float(c))
    return best[1], best[2], best[3]


def chinchilla_loss(n_params: float, n_tokens: float) -> float:
    return 1.69 + 406.4 / n_params ** 0.34 + 410.7 / n_tokens ** 0.28


def chinchilla_optimal(compute: float, tokens_per_param: float = 20.0) -> tuple[float, float]:
    n = math.sqrt(compute / (6.0 * tokens_per_param))
    return n, tokens_per_param * n

## 3. Thí nghiệm: loss theo kích thước model

Cùng dữ liệu và tokenizer với bài Intermediate 01, `n_layer = 4`, chỉ đổi `n_embd`; mọi model thấy **cùng số token**. Lớp
`GPT` nạp từ bài Basic 01 của bạn (chưa xong thì từ lời giải). Đây là một thí nghiệm nhỏ: 5 điểm, 1 seed, ngân sách rất
nhỏ, nên số mũ khớp được chỉ là ước lượng thô; xem mục 7 lý thuyết để so với các con số đã công bố.

In [ ]:
# explore: dữ liệu, tokenizer, lớp GPT
import importlib.util
import time
import unicodedata

import pyarrow.parquet as pq
import torch
import torch.nn.functional as F
from tokenizers import Tokenizer, decoders, models, pre_tokenizers, trainers

texts = pq.read_table(SHARD, columns=["text"]).column("text").to_pylist()
articles = [unicodedata.normalize("NFC", t) for t in texts if len(t) >= 2000]
perm = np.random.RandomState(0).permutation(len(articles))
train_text = "\n\n".join(articles[i] for i in perm[:1000])
val_text = "\n\n".join(articles[i] for i in perm[1000:1100])
tok = Tokenizer(models.BPE())
tok.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tok.decoder = decoders.ByteLevel()
tok.train_from_iterator([train_text], trainers.BpeTrainer(vocab_size=8000, initial_alphabet=pre_tokenizers.ByteLevel.alphabet()))
train_ids = torch.tensor(tok.encode(train_text).ids)
val_ids = torch.tensor(tok.encode(val_text).ids)


def load_gpt_module():
    base = ROOT / "tracks/ai-engineer/modules/05-transformers-llm-architecture"
    for path in [base / "exercises/basic/01-gpt-mini/starter/gpt.py", base / "solutions/basic/01-gpt-mini/gpt.py"]:
        spec = importlib.util.spec_from_file_location("gpt_mini", path)
        module = importlib.util.module_from_spec(spec)
        sys.modules["gpt_mini"] = module
        spec.loader.exec_module(module)
        try:
            module.GPT(module.GPTConfig(vocab_size=10, block_size=4, n_layer=1, n_head=1, n_embd=8))(torch.zeros(1, 2, dtype=torch.long))
            print("dùng", path.relative_to(ROOT))
            return module
        except NotImplementedError:
            continue


gpt_mini = load_gpt_module()

In [ ]:
# explore: train 5 model cùng số token (CPU khoảng 5 phút)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(4)
if DEVICE == "cuda":
    WIDTHS, STEPS, BLOCK = [64, 128, 192, 256, 384], 2000, 128
else:
    WIDTHS, STEPS, BLOCK = [32, 64, 96, 128, 192], 600, 64
BATCH, N_LAYER = 32, 4
val_eval = val_ids[:(min(len(val_ids) - 1, 50_000) // BLOCK) * BLOCK + 1]


@torch.no_grad()
def val_loss(model):
    model.eval()
    x = val_eval[:-1].view(-1, BLOCK)
    y = val_eval[1:].view(-1, BLOCK)
    total = sum(F.cross_entropy(model(x[i:i + 64].to(DEVICE))[0].float().transpose(1, 2), y[i:i + 64].to(DEVICE),
                                reduction="sum").item() for i in range(0, len(x), 64))
    model.train()
    return total / y.numel()


results = []
for width in WIDTHS:
    torch.manual_seed(0)
    cfg = gpt_mini.GPTConfig(vocab_size=8000, block_size=BLOCK, n_layer=N_LAYER, n_head=4, n_embd=width)
    model = gpt_mini.GPT(cfg).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=2e-3 if width <= 128 else 1e-3, betas=(0.9, 0.95), weight_decay=0.1)
    g, curve, start = torch.Generator().manual_seed(0), [], time.perf_counter()
    for step in range(STEPS):
        for group in opt.param_groups:
            group["lr"] = group.setdefault("max_lr", group["lr"]) * min(1.0, (step + 1) / 50) * (0.1 + 0.45 * (1 + math.cos(math.pi * step / STEPS)))
        starts = torch.randint(len(train_ids) - BLOCK, (BATCH,), generator=g)
        x = torch.stack([train_ids[s:s + BLOCK] for s in starts]).to(DEVICE)
        y = torch.stack([train_ids[s + 1:s + BLOCK + 1] for s in starts]).to(DEVICE)
        with torch.autocast("cuda", dtype=torch.bfloat16, enabled=DEVICE == "cuda"):
            _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
        curve.append(loss.item())
    n = gpt_params(N_LAYER, width, 8000, BLOCK)["non_embedding"]
    results.append(dict(width=width, n=n, total=model.num_params(), loss=val_loss(model), curve=curve,
                        seconds=time.perf_counter() - start))
    print(f"n_embd {width:4d}: {n:>10,} tham số non-embedding, val loss {results[-1]['loss']:.3f}, {results[-1]['seconds']:.0f} s")
tokens = STEPS * BATCH * BLOCK
print(f"mỗi model thấy {tokens:,} token")

In [ ]:
# explore: khớp L(N) = c + a N^-alpha, vẽ loss theo compute
import matplotlib.pyplot as plt

N = np.array([r["n"] for r in results], dtype=float)
L = np.array([r["loss"] for r in results])
a, alpha, c = fit_power_law(N, L)
print(f"L(N) ≈ {c:.2f} + {a:.1f} · N^-{alpha:.3f}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
grid = np.geomspace(N.min() / 2, N.max() * 4, 100)
ax[0].scatter(N, L, zorder=3)
ax[0].plot(grid, c + a * grid ** -alpha, "--", label=f"c + a·N^-{alpha:.2f}")
ax[0].set_xscale("log"); ax[0].set_xlabel("tham số non-embedding N"); ax[0].set_ylabel("val loss (nat/token)")
ax[0].legend(); ax[0].set_title("Loss theo kích thước, cùng số token")
smooth = lambda v: np.convolve(v, np.ones(20) / 20, mode="valid")
for r in results:
    per_step = train_flops(r["n"], BATCH * BLOCK)
    curve = smooth(r["curve"])
    ax[1].plot(per_step * np.arange(20, 20 + len(curve)), curve, label=f"n_embd {r['width']}")
ax[1].set_xscale("log"); ax[1].set_xlabel("compute (FLOP, 6ND)"); ax[1].set_ylabel("loss train"); ax[1].legend(fontsize=8)
ax[1].set_title("Loss theo compute")
plt.tight_layout(); plt.show()

In [ ]:
# explore: Chinchilla: ngân sách của Gopher, chia thế nào cho tối ưu
gopher = train_flops(280e9, 300e9)
n_opt, d_opt = chinchilla_optimal(gopher)
print(f"compute của Gopher: {gopher:.2e} FLOP")
print(f"Chinchilla khuyên: {n_opt / 1e9:.0f} tỉ tham số, {d_opt / 1e12:.2f} nghìn tỉ token")
for name, n, d in [("Gopher 280B, 300B token", 280e9, 300e9), ("Chinchilla 70B, 1.4T token", 70e9, 1.4e12)]:
    print(f"{name:28s}: L = {chinchilla_loss(n, d):.3f}")

## 4. Đọc kết quả

So với mục 7 phần lý thuyết.

- Trong khoảng đo, loss giảm gần như **thẳng** theo $$\log N$$: mỗi lần nhân đôi số tham số bớt một lượng loss gần như cố
  định. Chưa thấy đường cong đi ngang, nên $$c$$ (loss không giảm được) không xác định được từ 5 điểm: hàm khớp đẩy $$c$$ về
  gần 0 hoặc một giá trị tuỳ ý, còn số mũ chỉ là ước lượng thô. Đừng kết luận gì từ chữ số thứ hai.
- Trên biểu đồ loss theo compute, ở **cùng một mức compute** model nhỏ có loss thấp hơn, và các đường không cắt nhau trong
  ngân sách này: model lớn chỉ tốt hơn vì được tiêu nhiều compute hơn. Tự tính số token mỗi tham số của từng model và so với mức
  Chinchilla: model lớn được train quá ít token so với kích thước, nên ở ngân sách này ta đang ở phía compute nhỏ của
  "đường bao", nơi model nhỏ hơn là lựa chọn tối ưu. Chinchilla đo đúng
  đường bao này ở quy mô lớn và ra quy tắc khoảng 20 token mỗi tham số.
- Với ngân sách của Gopher, công thức Chinchilla cho loss thấp hơn khi dùng model nhỏ hơn 4 lần, train trên gần 5 lần dữ
  liệu. Llama 3 8B train trên 15 nghìn tỉ token (gần 1 900 token mỗi tham số): khi tính cả chi phí **inference**, model nhỏ
  train lâu hơn mức Chinchilla mới là lựa chọn rẻ.

Sau khi chạy hết notebook, làm phần **trắc nghiệm** của bài trong app.